In [1]:
from pydantic import BaseModel, ConfigDict, Field
from typing import Dict, List, Literal, Optional, Set, Any
from langchain.output_parsers import PydanticOutputParser
import json
import pandas as pd

In [2]:
CHOICES = ["A", "B", "C", "D"]

class MCQResponse(BaseModel):
    answer: Literal["A", "B", "C", "D"] = Field(
        ..., title="The answer to the multiple choice question"
    )
    rationale: str = Field(..., title="The rationale for the answer")

    @property
    def answer_index(self):
        return CHOICES.index(self.answer)

class MCQCvarChoice(BaseModel):
    cvar_str: str
    rationale: str
    is_llm_generated: bool

    def __hash__(self) -> int:
        return hash(self.cvar_str + self.rationale)
        
class MCQTransformChoice(BaseModel):
    code: str
    rationale: str
    is_llm_generated: bool

    def __hash__(self) -> int:
        return hash(self.code + self.rationale)


class MCQSimpleCvar(BaseModel):
    mc_type: Literal["select_pos", "select_neg"]
    options: List[MCQCvarChoice]
    correct_answer: MCQCvarChoice

    def __str__(self):
        s = f"MCQ: {self.mc_type}\n"
        for i, option in enumerate(self.options):
            s += f"Option {i+1}: {option.cvar_str}\n"
            s += f"Rationale: {option.rationale}\n"
            s += "----\n"
        s += f"Correct Answer: {self.correct_answer.cvar_str}\n"
        return s

    @property
    def task_instruction(self):
        s = "MOST" if self.mc_type == "select_pos" else "LEAST"
        instr = f"""Given the research question and dataset, we want to perform an analysis to answer the question.
Of the choices below, select the conceptual variable that is {s} justifiable for the analysis.
"""
        return instr

    @property
    def valid_values(self):
        return CHOICES[: len(self.options)]

    @property
    def choices(self):

        s = ""
        for i, opt in enumerate(self.options):
            s += f"{CHOICES[i]}. {opt.cvar_str}\n"
        return s
class MCQSimpleTransform(BaseModel):
    coneptual_var_str: str
    mc_type: Literal["select_pos", "select_neg"]
    options: List[MCQTransformChoice]
    correct_answer: MCQTransformChoice

    @property
    def task_instruction(self):
        s = "MOST" if self.mc_type == "select_pos" else "LEAST"
        instr = f"""Given the research question and dataset, we want to perform an analysis to answer the question. 
Specifically we want to operationalize the conceptual variable *{self.coneptual_var_str.lower()}* which we will use for statistical modeling. 
Of the choices given, select transformation code that is {s} justifiable to operationalize *{self.coneptual_var_str.lower()}*."
"""
        return instr

    @property
    def valid_values(self):
        return CHOICES[: len(self.options)]

    @property
    def choices(self):
        s = ""
        for i, opt in enumerate(self.options):
            s += f"{CHOICES[i]}.\n```python\n{opt.code}\n```\n"
        return s

    def __str__(self):
        s = f"MCQ Transform: {self.mc_type} for operationalizing {self.coneptual_var_str}\n"
        for i, option in enumerate(self.options):
            s += f"Option {i+1}: {option.code}\n"
            s += f"Rationale: {option.rationale}\n"
            s += "----\n"
        s += f"Correct Answer: {self.options.index(self.correct_answer) + 1}\n"
        return s


class MCQDatasetSimple(BaseModel):
    mcqs_cvar: List[MCQSimpleCvar]
    mcqs_transform: Dict[str, List[MCQSimpleTransform]]

    @property
    def num_cvars(self):
        return len(self.mcqs_cvar)

    @property
    def num_transforms(self):
        return sum([len(v) for v in self.mcqs_transform.values()])

    @property
    def num_mcqs(self):
        return self.num_cvars + self.num_transforms
    
    @property
    def expected_correct(self):
        count = 0
        for mcq in self.mcqs_cvar:
            count += 1 / len(mcq.options)
        for k, mcqs in self.mcqs_transform.items():
            for mcq in mcqs:
                count += 1 / len(mcq.options)
        return count
    
class DatasetInfo(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)
    research_questions: List[str]
    data_desc: Optional[Dict[str, Any]] = None
    df: Optional[pd.DataFrame] = None

    @property
    def research_question(self):
        return self.research_questions[0]

    @property
    def data_desc_no_desc(self):
        if self.data_desc is None:
            return None
        ret = {**self.data_desc}
        ret.pop("dataset_description", None)
        return ret

    @property
    def data_desc_no_semantic_type(self):
        if self.data_desc is None:
            return None
        ret = {**self.data_desc}
        for f in ret["fields"]:
            f["properties"].pop("semantic_type", None)
        return ret

    @property
    def data_desc_no_desc_no_semantic_type(self):
        if self.data_desc is None:
            return None
        ret = {**self.data_desc}
        ret.pop("dataset_description", None)
        for f in ret["fields"]:
            f["properties"].pop("semantic_type", None)
        return ret



In [17]:
INSTRUCTION_PROMPT = """<Instruction>
{task_instruction}
In addition to the answer please also include a rationale.
Return your answer in the format specified below:
{format_instructions}
</Instruction> 

Research Question: {research_question}
Dataset: {dataset}

{mcq_choices}
The valid values are: {valid_values}
Answer: """

i = 0
pref = 'BLAD'
BLADE_res = []



In [18]:
# process mortgage
path = '/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/mortgage/mcq_dataset.json'
MCQ_data = MCQDatasetSimple(**json.load(open(path)))
data_path = "/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/mortgage/info.json"

dinfo = DatasetInfo(**json.load(open(data_path)))

data_desc = json.dumps(dinfo.data_desc, indent=2)
parser = PydanticOutputParser(pydantic_object=MCQResponse)


data_file = data_path.split('/datasets/')[1]
for k,v in MCQ_data.mcqs_transform.items():
    # print(k,v)
    for mcq in v:
        prompt_template = INSTRUCTION_PROMPT
        prompt_variables = {
                "research_question": dinfo.research_question,
                "dataset": data_desc,
                "task_instruction": mcq.task_instruction,
                "mcq_choices": mcq.choices,
                "format_instructions": parser.get_format_instructions(),
                "valid_values": ", ".join(mcq.valid_values),
            }
        prompt = prompt_template.format(**prompt_variables)
        answer = CHOICES[mcq.options.index(mcq.correct_answer)]
        tem_ques = {
                'id': pref+'_'+str(i),
                'question': prompt,
                'data_file': data_file,  
                'doc_file': 'None',  
                'answer': answer, 
                'data_domain': 'Finance',  # The domain the data belongs to (e.g., finance, education)
                'analysis_type': "Structure problems",  # The type of question, optional: ["Structure problems", "Unstructured problems", "Chart problems"]
                'origin_from': ['DSBench',data_file.split('/')[0]],  # Source of the question, e.g., ['benchmark name', 'question id']
                'additional_information': '',  # Additional information such as code, statistical results, intermediate steps (e.g., StatQA's analysis methods) etc.
            }
        BLADE_res.append(tem_ques)
        i+=1


In [19]:
# process mortgage
path = '/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/reading/mcq_dataset.json'
MCQ_data = MCQDatasetSimple(**json.load(open(path)))
data_path = "/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/reading/info.json"

dinfo = DatasetInfo(**json.load(open(data_path)))

data_desc = json.dumps(dinfo.data_desc, indent=2)
parser = PydanticOutputParser(pydantic_object=MCQResponse)


data_file = data_path.split('/datasets/')[1]
for k,v in MCQ_data.mcqs_transform.items():
    # print(k,v)
    for mcq in v:
        prompt_template = INSTRUCTION_PROMPT
        prompt_variables = {
                "research_question": dinfo.research_question,
                "dataset": data_desc,
                "task_instruction": mcq.task_instruction,
                "mcq_choices": mcq.choices,
                "format_instructions": parser.get_format_instructions(),
                "valid_values": ", ".join(mcq.valid_values),
            }
        prompt = prompt_template.format(**prompt_variables)
        answer = CHOICES[mcq.options.index(mcq.correct_answer)]
        tem_ques = {
                'id': pref+'_'+str(i),
                'question': prompt,
                'data_file': data_file,  
                'doc_file': 'None',  
                'answer': answer, 
                'data_domain': 'Education',  # The domain the data belongs to (e.g., finance, education)
                'analysis_type': "Structure problems",  # The type of question, optional: ["Structure problems", "Unstructured problems", "Chart problems"]
                'origin_from': ['DSBench',data_file.split('/')[0]],  # Source of the question, e.g., ['benchmark name', 'question id']
                'additional_information': '',  # Additional information such as code, statistical results, intermediate steps (e.g., StatQA's analysis methods) etc.
            }
        BLADE_res.append(tem_ques)
        i+=1


In [21]:
# process mortgage
path = '/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/boxes/mcq_dataset.json'
MCQ_data = MCQDatasetSimple(**json.load(open(path)))
data_path = "/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/boxes/info.json"

dinfo = DatasetInfo(**json.load(open(data_path)))

data_desc = json.dumps(dinfo.data_desc, indent=2)
parser = PydanticOutputParser(pydantic_object=MCQResponse)


data_file = data_path.split('/datasets/')[1]
for k,v in MCQ_data.mcqs_transform.items():
    # print(k,v)
    for mcq in v:
        prompt_template = INSTRUCTION_PROMPT
        prompt_variables = {
                "research_question": dinfo.research_question,
                "dataset": data_desc,
                "task_instruction": mcq.task_instruction,
                "mcq_choices": mcq.choices,
                "format_instructions": parser.get_format_instructions(),
                "valid_values": ", ".join(mcq.valid_values),
            }
        prompt = prompt_template.format(**prompt_variables)
        answer = CHOICES[mcq.options.index(mcq.correct_answer)]
        tem_ques = {
                'id': pref+'_'+str(i),
                'question': prompt,
                'data_file': data_file,  
                'doc_file': 'None',  
                'answer': answer, 
                'data_domain': 'Education',  # The domain the data belongs to (e.g., finance, education)
                'analysis_type': "Structure problems",  # The type of question, optional: ["Structure problems", "Unstructured problems", "Chart problems"]
                'origin_from': ['DSBench',data_file.split('/')[0]],  # Source of the question, e.g., ['benchmark name', 'question id']
                'additional_information': '',  # Additional information such as code, statistical results, intermediate steps (e.g., StatQA's analysis methods) etc.
            }
        BLADE_res.append(tem_ques)
        i+=1


In [23]:
# process mortgage
path = '/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/teachingratings/mcq_dataset.json'
MCQ_data = MCQDatasetSimple(**json.load(open(path)))
data_path = "/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/teachingratings/info.json"

dinfo = DatasetInfo(**json.load(open(data_path)))

data_desc = json.dumps(dinfo.data_desc, indent=2)
parser = PydanticOutputParser(pydantic_object=MCQResponse)


data_file = data_path.split('/datasets/')[1]
for k,v in MCQ_data.mcqs_transform.items():
    # print(k,v)
    for mcq in v:
        prompt_template = INSTRUCTION_PROMPT
        prompt_variables = {
                "research_question": dinfo.research_question,
                "dataset": data_desc,
                "task_instruction": mcq.task_instruction,
                "mcq_choices": mcq.choices,
                "format_instructions": parser.get_format_instructions(),
                "valid_values": ", ".join(mcq.valid_values),
            }
        prompt = prompt_template.format(**prompt_variables)
        answer = CHOICES[mcq.options.index(mcq.correct_answer)]
        tem_ques = {
                'id': pref+'_'+str(i),
                'question': prompt,
                'data_file': data_file,  
                'doc_file': 'None',  
                'answer': answer, 
                'data_domain': 'Education',  # The domain the data belongs to (e.g., finance, education)
                'analysis_type': "Structure problems",  # The type of question, optional: ["Structure problems", "Unstructured problems", "Chart problems"]
                'origin_from': ['DSBench',data_file.split('/')[0]],  # Source of the question, e.g., ['benchmark name', 'question id']
                'additional_information': '',  # Additional information such as code, statistical results, intermediate steps (e.g., StatQA's analysis methods) etc.
            }
        BLADE_res.append(tem_ques)
        i+=1


In [25]:
# process mortgage
path = '/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/caschools/mcq_dataset.json'
MCQ_data = MCQDatasetSimple(**json.load(open(path)))
data_path = "/Users/zhangyingyi/Library/CloudStorage/OneDrive-CityUniversityofHongKong-Student/With_yingyi_AI4BI/code_new/BA-bench/BLADE/blade_bench/datasets/caschools/info.json"

dinfo = DatasetInfo(**json.load(open(data_path)))

data_desc = json.dumps(dinfo.data_desc, indent=2)
parser = PydanticOutputParser(pydantic_object=MCQResponse)


data_file = data_path.split('/datasets/')[1]
for k,v in MCQ_data.mcqs_transform.items():
    # print(k,v)
    for mcq in v:
        prompt_template = INSTRUCTION_PROMPT
        prompt_variables = {
                "research_question": dinfo.research_question,
                "dataset": data_desc,
                "task_instruction": mcq.task_instruction,
                "mcq_choices": mcq.choices,
                "format_instructions": parser.get_format_instructions(),
                "valid_values": ", ".join(mcq.valid_values),
            }
        prompt = prompt_template.format(**prompt_variables)
        answer = CHOICES[mcq.options.index(mcq.correct_answer)]
        tem_ques = {
                'id': pref+'_'+str(i),
                'question': prompt,
                'data_file': data_file,  
                'doc_file': 'None',  
                'answer': answer, 
                'data_domain': 'Education',  # The domain the data belongs to (e.g., finance, education)
                'analysis_type': "Structure problems",  # The type of question, optional: ["Structure problems", "Unstructured problems", "Chart problems"]
                'origin_from': ['DSBench',data_file.split('/')[0]],  # Source of the question, e.g., ['benchmark name', 'question id']
                'additional_information': '',  # Additional information such as code, statistical results, intermediate steps (e.g., StatQA's analysis methods) etc.
            }
        BLADE_res.append(tem_ques)
        i+=1


In [28]:
BLADE_res[-1]

{'id': 'BLAD_71',
 'question': '<Instruction>\nGiven the research question and dataset, we want to perform an analysis to answer the question. \nSpecifically we want to operationalize the conceptual variable *student\'s test scores* which we will use for statistical modeling. \nOf the choices given, select transformation code that is LEAST justifiable to operationalize *student\'s test scores*."\n\nIn addition to the answer please also include a rationale.\nReturn your answer in the format specified below:\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"answer": {"e